In [1]:
import sys 
sys.path.append("..")
import numpy as np
import pandas as pd
from src.winners_curse import split_halves

df = pd.read_csv("../data/clean/upworthy-archive-confirmatory-packages-clean.csv")
clean = df[df["problem"] == 0].copy()


/var/folders/n7/ygxn27wd1t1_v6rsct9c466w0000gn/T/ipykernel_23437/654407397.py:7: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/clean/upworthy-archive-confirmatory-packages-clean.csv")


## Confirmatory run
Same method as notebook 03, run on the held out confirmatory tests (17,836 tests after dropping the flagged window). Nothing in the method was changed.

In [3]:
rng = np.random.default_rng(2) #fixed seed, same splits each run
R = 100 # number of random split, its a placeholder to check it is stable
tid = clean["clickability_test_id"].to_numpy() # test id

rows = []
for r in range(R):
    c_sel, n_sel, c_val, n_val = split_halves(clean["clicks"], clean["impressions"], rng)
    d = pd.DataFrame({"test": tid, "sel": c_sel / n_sel, "val": c_val / n_val})
    d = d.sort_values(["test", "sel"], ascending=[True, False], kind = "stable")
    d["rank"] = d.groupby("test").cumcount()
    best = d[d["rank"] == 0].set_index("test")
    runner = d[d["rank"] == 1].set_index("test")
    # lead o f best over runner up in each half, for the SAME two packages
    rows.append(pd.DataFrame({
        "lead_sel": best["sel"] - runner["sel"],
        "lead_val": best["val"] - runner["val"],
        "rep": r,}))
leads = pd.concat(rows)
curse = leads["lead_sel"] - leads["lead_val"]

print(f"mean lead, selection half:  {leads['lead_sel'].mean() * 100:.3f} pp")
print(f"mean lead, validation half: {leads['lead_val'].mean() * 100:.3f} pp")
print(f"average winner's curse:     {curse.mean() * 100:.3f} pp")
    




mean lead, selection half:  0.324 pp
mean lead, validation half: 0.162 pp
average winner's curse:     0.162 pp


In [5]:

# one split, one test
c_sel, n_sel, c_val, n_val = split_halves(clean["clicks"], clean["impressions"], np.random.default_rng(0))
print(clean[["clicks", "impressions"]].head(4).assign(c_sel=c_sel[:4], n_sel=n_sel[:4], c_val=c_val[:4], n_val=n_val[:4]))


   clicks  impressions  c_sel  n_sel  c_val  n_val
0       8         3118      3   1559      5   1559
1     130         4587     70   2293     60   2294
2      19         3017     11   1508      8   1509
3      26         2974     17   1487      9   1487


Results on the confirmatory tests (17,836 tests), seed 6, 200 splits:
 - Selection half: best package leads the runner up by 0.324 pp on average
 - Validation half, same two packages: lead drops to 0.162 pp
 - Winner's curse: 0.163 pp, about half of the selection-half lead
 - 95% range over tests: 0.160 to 0.166 pp
 - Exploratory (notebook 03): 0.158 pp, range 0.152 to 0.164

To keep in mind:
 - Each half has half the visitors, so on full-size tests the curse is probably smaller.
 - The same date window was applied to both files; not checked against the archive documentation for this file.
 - The first run in this notebook (seed 2, 100 splits) gave 0.162 pp.

In [7]:
rng = np.random.default_rng(6) #fixed seed, same splits each run
R = 200 # number of random split, its a placeholder to check it is stable
tid = clean["clickability_test_id"].to_numpy() # test id

rows = []
for r in range(R):
    c_sel, n_sel, c_val, n_val = split_halves(clean["clicks"], clean["impressions"], rng)
    d = pd.DataFrame({"test": tid, "sel": c_sel / n_sel, "val": c_val / n_val})
    d = d.sort_values(["test", "sel"], ascending=[True, False], kind = "stable")
    d["rank"] = d.groupby("test").cumcount()
    best = d[d["rank"] == 0].set_index("test")
    runner = d[d["rank"] == 1].set_index("test")
    # lead o f best over runner up in each half, for the SAME two packages
    rows.append(pd.DataFrame({
        "lead_sel": best["sel"] - runner["sel"],
        "lead_val": best["val"] - runner["val"],
        "rep": r,}))
leads = pd.concat(rows)
curse = leads["lead_sel"] - leads["lead_val"]

print(f"mean lead, selection half:  {leads['lead_sel'].mean() * 100:.3f} pp")
print(f"mean lead, validation half: {leads['lead_val'].mean() * 100:.3f} pp")
print(f"average winner's curse:     {curse.mean() * 100:.3f} pp")

mean lead, selection half:  0.324 pp
mean lead, validation half: 0.162 pp
average winner's curse:     0.163 pp


In [8]:
# So, how much would the avg. curse (which is now 0.158 pp) would change if we had a different set of tests?
# lets see
boot_rng = np.random.default_rng(7)
B = 2000 # number of redraws
per_test = leads.groupby(level=0)[["lead_sel", "lead_val"]].mean()
curse = (per_test["lead_sel"] - per_test["lead_val"]).to_numpy()
print("tests:", len(curse))

boot = np.array([
    boot_rng.choice(curse, size=len(curse), replace = True).mean() # we redraw one curse number per test, 3,823 in total. and get pretend curse's mean 
    for _ in range(B)
])
low, high = np.percentile(boot, [2.5, 97.5]) * 100 # middle 95% - cut off low and high 2.5% amd convert to %
print(f"mean curse: {curse.mean() * 100:.3f} pp")
print(f"95% range:  [{low:.3f}, {high:.3f}] pp")

tests: 17836
mean curse: 0.163 pp
95% range:  [0.160, 0.166] pp
